# Number of parents adjacent to leaves in subject metadata
Count parent nodes in the subject metadata tree that are directly adjacent to at least one leaf node.

In [3]:
import csv
from collections import defaultdict
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "metadata" / "subject_metadata.csv").exists():
            return cand
    raise FileNotFoundError("Could not find project root containing data/metadata/subject_metadata.csv")


root = find_project_root()
subject_csv = root / "data" / "metadata" / "subject_metadata.csv"

parent_by_child: dict[int, int] = {}
children_by_parent: dict[int, set[int]] = defaultdict(set)
all_subject_ids: set[int] = set()

with subject_csv.open("r", encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        sid = int(row["SubjectId"])
        all_subject_ids.add(sid)

        p = row.get("ParentId", "")
        if p and p != "NULL":
            pid = int(p)
            parent_by_child[sid] = pid
            children_by_parent[pid].add(sid)

leaf_subject_ids = all_subject_ids - set(children_by_parent.keys())

# Parent nodes adjacent to leaves = parents of leaf nodes
parents_adjacent_to_leaves = {parent_by_child[sid] for sid in leaf_subject_ids if sid in parent_by_child}

# Optional detail: number of leaf children per such parent
leaf_children_count_by_parent = {
    pid: sum(1 for child in children_by_parent.get(pid, set()) if child in leaf_subject_ids)
    for pid in parents_adjacent_to_leaves
}

print(f"Total subject nodes: {len(all_subject_ids):,}")
print(f"Leaf nodes: {len(leaf_subject_ids):,}")
print(f"Parent nodes adjacent to a leaf: {len(parents_adjacent_to_leaves):,}")

print("\nTop 20 parents by number of leaf children:")
for pid, c in sorted(leaf_children_count_by_parent.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  Parent {pid}: {c} leaf children")

Total subject nodes: 388
Leaf nodes: 316
Parent nodes adjacent to a leaf: 69

Top 20 parents by number of leaf children:
  Parent 342: 14 leaf children
  Parent 144: 13 leaf children
  Parent 54: 11 leaf children
  Parent 39: 10 leaf children
  Parent 74: 9 leaf children
  Parent 348: 9 leaf children
  Parent 126: 9 leaf children
  Parent 154: 8 leaf children
  Parent 98: 8 leaf children
  Parent 46: 7 leaf children
  Parent 338: 7 leaf children
  Parent 141: 6 leaf children
  Parent 146: 6 leaf children
  Parent 278: 6 leaf children
  Parent 158: 6 leaf children
  Parent 174: 6 leaf children
  Parent 55: 6 leaf children
  Parent 62: 6 leaf children
  Parent 81: 6 leaf children
  Parent 97: 6 leaf children
